In [2]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, f1_score, classification_report, mean_absolute_error, root_mean_squared_error
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier, XGBRegressor

SEED = 42
np.random.seed(SEED)

In [3]:
df = pd.read_csv('./data/bins.csv', parse_dates=['date'])
print(f"Shape: {df.shape}  |  Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Contamination prevalence: {df['is_contaminated'].mean():.1%}")

DROP_COLS = ['date', 'bin_id', 'real_base_contam', 'commodity_prior', 'contamination_rate', 'is_contaminated']
FEATURES = [c for c in df.columns if c not in DROP_COLS]
TARGET_CLASS, TARGET_REG = 'is_contaminated', 'contamination_rate'

dates = df['date'].sort_values().unique()
n = len(dates)
train_end = dates[int(n * 0.70)]
val_end   = dates[int(n * 0.85)]

train = df[df['date'] <  train_end]
val   = df[(df['date'] >= train_end) & (df['date'] < val_end)]
test  = df[df['date'] >= val_end]

X_train, y_train_c, y_train_r = train[FEATURES], train[TARGET_CLASS], train[TARGET_REG]
X_val,   y_val_c,   y_val_r   = val[FEATURES],   val[TARGET_CLASS],   val[TARGET_REG]
X_test,  y_test_c,  y_test_r  = test[FEATURES],  test[TARGET_CLASS],  test[TARGET_REG]

print(f"Train {len(train):,} | Val {len(val):,} | Test {len(test):,}")

Shape: (27010, 30)  |  Date range: 2018-01-01 -> 2018-12-31
Contamination prevalence: 49.0%
Train 18,870 | Val 4,070 | Test 4,070


In [4]:
scale_pos = (y_train_c == 0).sum() / (y_train_c == 1).sum()

xgb_clf = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos,
    eval_metric='auc', early_stopping_rounds=30, random_state=SEED, verbosity=0
)
xgb_clf.fit(X_train, y_train_c, eval_set=[(X_val, y_val_c)], verbose=False)

xgb_reg = XGBRegressor(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='rmse', early_stopping_rounds=30, random_state=SEED, verbosity=0
)
xgb_reg.fit(X_train, y_train_r, eval_set=[(X_val, y_val_r)], verbose=False)

xgb_val_prob = xgb_clf.predict_proba(X_val)[:, 1]
xgb_rate_val = np.clip(xgb_reg.predict(X_val), 0, 1)

print(f"Classifier -- Val AUC: {roc_auc_score(y_val_c, xgb_val_prob):.4f}  F1: {f1_score(y_val_c, xgb_clf.predict(X_val)):.4f}")
print(f"Regressor  -- Val MAE: {mean_absolute_error(y_val_r, xgb_rate_val):.4f}  RMSE: {root_mean_squared_error(y_val_r, xgb_rate_val):.4f}")

Classifier -- Val AUC: 0.9089  F1: 0.8576
Regressor  -- Val MAE: 0.0320  RMSE: 0.0399


In [5]:
ir = IsotonicRegression(out_of_bounds='clip')
ir.fit(xgb_val_prob, y_val_c)

class _IsotonicCalibrator:
    def __init__(self, base_clf, ir):
        self.base_clf, self.ir = base_clf, ir
    def predict_proba(self, X):
        raw = self.base_clf.predict_proba(X)[:, 1]
        cal = np.clip(self.ir.predict(raw), 0, 1)
        return np.column_stack([1 - cal, cal])

cal_xgb = _IsotonicCalibrator(xgb_clf, ir)
cal_val_prob = cal_xgb.predict_proba(X_val)[:, 1]
print(f"Val AUC after calibration: {roc_auc_score(y_val_c, cal_val_prob):.4f}")

Val AUC after calibration: 0.9113


In [6]:
test_prob = cal_xgb.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)
test_rate = np.clip(xgb_reg.predict(X_test), 0, 1)

print("TEST SET")
print(f"AUC-ROC: {roc_auc_score(y_test_c, test_prob):.4f}  |  F1: {f1_score(y_test_c, test_pred):.4f}")
print(f"Rate MAE: {mean_absolute_error(y_test_r, test_rate):.4f}  |  RMSE: {root_mean_squared_error(y_test_r, test_rate):.4f}")
print()
print(classification_report(y_test_c, test_pred, target_names=['Clean', 'Contaminated']))

TEST SET
AUC-ROC: 0.9118  |  F1: 0.8556
Rate MAE: 0.0331  |  RMSE: 0.0413

              precision    recall  f1-score   support

       Clean       0.79      0.82      0.80      1684
Contaminated       0.87      0.84      0.86      2386

    accuracy                           0.83      4070
   macro avg       0.83      0.83      0.83      4070
weighted avg       0.83      0.83      0.83      4070



In [7]:
X_all = df[FEATURES]
all_prob = np.clip(cal_xgb.predict_proba(X_all)[:, 1], 0, 1)
all_rate = np.clip(xgb_reg.predict(X_all), 0, 1)
loc_cols = [c for c in FEATURES if c.startswith('loc_')]
location_type = df[loc_cols].idxmax(axis=1).str.replace('loc_', '')

output = pd.DataFrame({
    'date':                      df['date'].dt.strftime('%Y-%m-%d'),
    'bin_id':                    df['bin_id'],
    'location_type':             location_type,
    'bin_volume_g':              df['bin_volume_g'],
    'capacity_fill':             df['capacity_fill'],
    'predicted_contam_prob':     all_prob.round(4),
    'predicted_contam_rate':     all_rate.round(4),
    'predicted_is_contaminated': (all_prob >= 0.5).astype(int),
})
output.to_csv('./data/contamination_predictions.csv', index=False)
print(f"Saved to ./data/contamination_predictions.csv")

summary = output.groupby('location_type').agg(
    avg_contam_prob=('predicted_contam_prob', 'mean'),
    pct_contaminated=('predicted_is_contaminated', 'mean')
).round(3).sort_values('avg_contam_prob', ascending=False)

print(summary.to_string())

Saved to ./data/contamination_predictions.csv
               avg_contam_prob  pct_contaminated
location_type                                   
dormitory                0.830             0.882
dining_hall              0.682             0.766
student_union            0.569             0.464
gym                      0.383             0.280
academic_bldg            0.176             0.072
library                  0.064             0.003
